# 🌳 Tree of Thought + Subquestion Decomposition

**Welcome back!** In this notebook, you'll explore two more sophisticated prompting strategies that go beyond simple Chain of Thought.

By the end, you'll have built a working **Tree of Thought** story writer and implemented **Subquestion Decomposition** and **Plan-and-Solve** prompting — both from scratch.

---

**How to use this notebook:**
- Cells marked **[RUN]** — just execute them, no changes needed.
- Cells marked **[TODO]** — fill in the missing code before running.

## 1. Setup the environment and define utility functions

**[RUN]** Install dependencies and load the Qwen model. No edits needed here!

In [ ]:
#@title Install Dependencies {display-mode: "form"}
#@markdown Run this cell to install the required packages.
!pip install transformers
!pip install Jinja2==3.1.6
!pip install accelerate

In [ ]:
#@title Load the Model and Tokenizer {display-mode: "form"}
#@markdown Loads **Qwen2.5-0.5B-Instruct** and its tokenizer. This may take a few minutes on the first run.
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", 
                                             device_map="auto",
                                             dtype=torch.float16 if torch.cuda.is_available() else torch.float32)

In [ ]:
#@title Utility and Helper Functions {display-mode: "form"}
#@markdown Helper functions to generate model response and display results.
import uuid
import time
from IPython.display import display, HTML, Javascript
import html as html_lib

# --- Model generation utility ---
def generate_model_response(prompt, max_new_tokens=256):
    """
    Runs the model on a given prompt and returns the response text.
    """
    start_time = time.time()
    inputs = tokenizer.apply_chat_template(
        prompt,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

    elapsed = time.time() - start_time
    return response, elapsed

# --- Display utility ---
def display_model_interaction(prompt):
    """
    Displays the prompt and model response in a styled table,
    with a loading indicator while the model generates text.
    """
    uid = str(uuid.uuid4()).replace("-", "")

    # HTML scaffold (before generation)
    html = f"""
    <style>
    .prompt-table {{
        border-collapse: collapse;
        width: 100%;
        margin: 12px 0;
        font-family: 'Segoe UI', sans-serif;
    }}
    .prompt-table th, .prompt-table td {{
        border: 1px solid #ccc;
        padding: 10px;
        vertical-align: top;
    }}
    .prompt-table th {{
        background-color: #f2f2f2;
        width: 20%;
    }}
    .loading {{
        color: #888;
        font-style: italic;
        animation: pulse 1.5s infinite;
    }}
    @keyframes pulse {{
        0% {{ opacity: 0.3; }}
        50% {{ opacity: 1; }}
        100% {{ opacity: 0.3; }}
    }}
    </style>
    <table class="prompt-table">
        <tr><th>Prompt</th><td>{prompt}</td></tr>
        <tr><th>Model Response</th><td id="response_{uid}">
            <span class="loading">⏳ Generating response...</span>
        </td></tr>
    </table>
    """

    # Display initial table
    display(HTML(html))

    # --- Call the model separately ---
    response_text, elapsed = generate_model_response(prompt)
    # print(response_text)
    # Escape special chars for HTML display
    safe_response = html_lib.escape(response_text.strip())

    # --- Inject response dynamically ---
    js = Javascript(f"""
        document.getElementById("response_{uid}").innerHTML =
            `<pre style="white-space: pre-wrap;">{safe_response}</pre>
             <div style='color:#666; font-size:90%; margin-top:4px;'>⏱ Generated in {elapsed:.2f} seconds</div>`;
    """)
    display(js)

def display_sample(question, answer):
    display(HTML(
        f"""<p>Question/answer 0:</p>
        <strong>Question: </strong>{question}
        <p><strong>Answer: </strong>{answer}</p>
        """
    ))

## 🌳 Part 1: Tree of Thought (ToT)

### What's different about Tree of Thought?

Standard Chain of Thought forces the model down a **single reasoning path**. But what if the first step it takes is suboptimal?

**Tree of Thought** lets the model branch out 🌳 — it generates *multiple candidate next steps*, scores them, picks the best, and keeps going. It's essentially **beam search applied to reasoning**.

Think of it like brainstorming: instead of committing to the first idea, you generate several options, compare them, and develop the most promising one.

Each section below walks through one stage of this process — from setup 🛠️ to proposal 💡 to final composition.

### Step 1: Configuration & Prompt Templates

**[RUN]** We define two key parameters:
- **B** (branching factor): how many candidate ideas to generate at each step
- **D** (depth): how many reasoning steps to take

🧮 Wider B → more ideas explored; deeper D → longer reasoning chains.

In [ ]:
from typing import List, Tuple
import random

# --- Config ---
SEED = 42
random.seed(SEED)

# Beam search / ToT parameters
B = 2   # branching factor: candidates per step
D = 2   # tree depth: outline beams (e.g., beginning / middle / end)

**[RUN]** Below are the three prompt templates that power the ToT pipeline:

- **`TASK_INSTRUCTION`**: the creative brief the story must follow.
- **`PROPOSE_PROMPT_TEMPLATE`**: asks the model for one candidate next step.
- **`SCORE_PROMPT_TEMPLATE`**: asks the model to score a candidate as a single integer 0–10.

> 🎛️ **Optional:** Feel free to tweak the task instruction or scoring criteria and see how it changes the output!

In [ ]:
TASK_INSTRUCTION = (
    "Write a vivid, 100 word short story for a general audience. "
    "It should feature: a surprising twist, strong imagery, and an emotionally satisfying resolution. "
    "Avoid proper names and instead use descriptive nouns."
)

PROPOSE_PROMPT_TEMPLATE = """
You are helping outline a short story. The global writing brief is:

{task}

Given the current outline (beams so far):
{outline}

Propose ONE next beam in 1-2 sentences that continues the outline naturally.
Keep it concrete and evocative. Do not repeat earlier beams.
Return ONLY the next beam, no explanations.
""".strip()

SCORE_PROMPT_TEMPLATE = """
You are evaluating a candidate *next beam* for a short-story outline.

Global brief:
{task}

beams so far:
{outline}

Candidate next beam:
{candidate_prompt}

Score the candidate from 0 to 10 for:
- Coherence with beams so far
- Creativity (fresh but plausible)
- Fitness for the brief (tone, twist potential, resolution potential)

Return ONLY a single integer 0-10 (no words).
""".strip()


### Quick Python refresher: filling in prompt templates

In the prompts above, you'll notice placeholders like `{task}` and `{outline}`. These get filled in using Python's `.format()` method. Let's try it out on the propose template:

In [ ]:
prompt = PROPOSE_PROMPT_TEMPLATE.format(task=TASK_INSTRUCTION, outline="Testing")
print(prompt)

### 🎯 [TODO] Try it yourself!

Now do the same for `SCORE_PROMPT_TEMPLATE`. Notice it has **three** placeholders: `{task}`, `{outline}`, and `{candidate_prompt}`. Fill all three with some dummy text and print the result.

In [ ]:
#your code here
prompt = SCORE_PROMPT_TEMPLATE.format(task=TASK_INSTRUCTION, outline="Testing", candidate_prompt="more testing")
print(prompt)

Great! Now you know how to construct prompt strings from templates. You'll use this pattern throughout the notebook.

### Step 2: ToT Core Functions (propose → score)

**[IMPORTANT]** Before running the next cell, read this note on **Chat Templates**:

When using a modern instruction-tuned model like Qwen, you don't just send a raw string. Instead, messages are formatted as a list of dictionaries, each with:
- **`role`**: either `"user"` or `"assistant"`
- **`content`**: the actual text

```python
message = [{"role": "user", "content": "Your prompt here"}]
```

This format is called the **Chat Template** and it's how the model knows who is speaking.

### 🎯 [TODO] Implement `score_candidates`

In `score_candidates`: build the prompt using `SCORE_PROMPT_TEMPLATE`, create the message dict, and call the model.

In [ ]:
def propose_next_beams(outline_beams: List[str], b: int) -> List[str]:
    outline_text = "\n- ".join(map(str, outline_beams)) if outline_beams else "(none yet)"
    prompt = PROPOSE_PROMPT_TEMPLATE.format(task=TASK_INSTRUCTION, outline=outline_text)
    message = [{"role": "user", "content": prompt}]
    response, _ = generate_model_response(message, max_new_tokens=64)
    return response


def score_candidates(outline_beams: List[str], candidate_prompts: List[str]) -> List[Tuple[str, float]]:
    scored = []
    outline_text = "\n- ".join(map(str, outline_beams)) if outline_beams else "(none yet)"
    for cand_prompt in candidate_prompts:
        prompt = SCORE_PROMPT_TEMPLATE.format(task=TASK_INSTRUCTION, outline=outline_text, candidate_prompt=cand_prompt)
        message = [{"role": "user", "content": prompt}]
        try:
            resp = generate_model_response(message, max_new_tokens=32)[0].strip()
            # Extract a single integer 0-10; be robust to stray text
            digits = ''.join(ch for ch in resp if ch.isdigit())
            score = float(digits) if digits else 0.0
            score = max(0.0, min(10.0, score))
        except Exception as e:
            score = 0.0
        scored.append((cand_prompt, score))
    return scored

### 🎯 [TODO] Step 3: Beam Search over Outline Steps

This is the core of the Tree-of-Thought algorithm 🌳. It loops through reasoning steps:
1. **Propose** candidate thoughts using `propose_next_beams`
2. **Score & select** the top `B` outlines using `score_candidates`
3. **Expand** further until the outline is complete

Your task: fill in the two `TODO` lines to call the right functions.

In [ ]:
def beam_search_outline(b: int = B, d: int = D) -> List[str]:
    # Each state is (beams, avg_score)
    beams: List[Tuple[List[str], float]] = [([], 0.0)]
    for step in range(1, d + 1):
        new_beams: List[Tuple[List[str], float]] = []
        for beams, avg_score in beams:
            candidate_prompts = propose_next_beams(beams, b=b)
            scored = score_candidates(beams, candidate_prompts)
            for cand, s in scored:
                new_beams.append((beams + [cand], (avg_score * (step-1) + s) / step))
        # Keep the top B beams
        new_beams.sort(key=lambda x: x[1], reverse=True)
        beams = new_beams[:b]
        print(f"Step {step}: top avg score = {beams[0][1]:.2f}")
    return beams[0][0]  # best outline beams

### 🎯 [TODO] Step 4: Compose the Final Story

Once we have the best outline beams, we ask the model to write the actual story.

Complete the two `TODO` lines below:

In [ ]:
COMPOSE_PROMPT_TEMPLATE = """
Using the outline below, write the final story (100 words). Follow the brief closely.
Write in a single block of prose.

Brief:
{task}

Outline beams:
- {beams}

Begin the story now.
""".strip()

def compose_story_from_outline(beams: List[str]) -> str:
    beams_text = "\n- ".join(beams)
    prompt = COMPOSE_PROMPT_TEMPLATE.format(task=TASK_INSTRUCTION, beams=beams_text)
    message = [{"role": "user", "content": prompt}]
    display_model_interaction(message)

### Step 5: Run the Full ToT Pipeline

**[RUN]** Let's put it all together!

In [ ]:
best_outline = beam_search_outline(b=B, d=D)
print("\nBest outline beams:")
for i, beam in enumerate(best_outline, 1):
    print(f"{i}. {beam}")

print("\n--- Final Story ---\n")
compose_story_from_outline(best_outline)

### Baseline: Direct Zero-Shot Story

**[RUN]** Now let's ask the model to write the same story without any ToT scaffolding — just a simple zero-shot prompt. Compare the quality!

In [ ]:
BASELINE_PROMPT = """
{task}

Write the story now (100 words) in one block.
""".strip()

def baseline_story() -> str:
    prompt = BASELINE_PROMPT.format(task=TASK_INSTRUCTION)
    message = [{"role": "user", "content": prompt}]
    display_model_interaction(message)

baseline_story()

---

## 🧠 Part 2: Subquestion Decomposition

### The big idea

Sometimes a problem is just too complex to solve in one shot 🧠💥. Instead of attacking it head-on, we break it into smaller, focused sub-questions — each one answerable on its own — and combine the answers at the end.

This mimics how humans tackle hard problems: **think → split → solve → combine** 🔄

We'll test this on the **GSM8K socratic split**, which includes worked examples with explicit subquestion breakdowns.

**[RUN]** Load the socratic version of GSM8K and check out a sample.

In [ ]:
#@title Load the Socratic GSM8K Split {display-mode: "form"}
#@markdown Loads the socratic version of GSM8K, where each answer is broken into guided sub-questions.
train_split_socratic = load_dataset("openai/gsm8k", "socratic", split="train[:1]")
test_split_socratic = load_dataset("openai/gsm8k", "socratic", split="test[:1]")

print(f"Train split size: {len(train_split_socratic)}")
print(f"Test split size: {len(test_split_socratic)}")

In [ ]:
#@title Explore a Socratic Sample {display-mode: "form"}
#@markdown Displays the first question-answer pair from the socratic training split.
display_sample(train_split_socratic[0]['question'], train_split_socratic[0]['answer'])

### Few-shot Subquestion Decomposition

**[RUN]** Let's first show the model an example of decomposition (few-shot), then ask it to apply the same approach to a new question.

In [ ]:
#@title Build the Few-Shot Prompt {display-mode: "form"}
#@markdown Constructs a few-shot prompt using one training example as an exemplar, then appends a test question.
few_shot_prompt_socratic = []

for i in range(1):
    s = train_split_socratic[i]
    q = str(s["question"]).strip()
    a = str(s["answer"]).strip()
    few_shot_prompt_socratic.append(f"Q: {q}\nA: {a}")

test_q = str(test_split_socratic[0]["question"]).strip()
blocks = []
instruction = "Here is an example on how to solve the question:"
blocks.append(instruction.strip())
blocks.extend(few_shot_prompt_socratic)
blocks.append(f"Please follow the same approach to answer the following question:\nQ: {test_q}\n")

few_shot_soc = "\n\n".join(blocks)
print(few_shot_soc)

In [ ]:
#@title Run Few-Shot Subquestion Decomposition {display-mode: "form"}
#@markdown Sends the few-shot prompt to the model and displays the response.
message = [{"role": "user", "content": few_shot_soc}]
display_model_interaction(message)

### 🎯 [TODO] Zero-shot Subquestion Decomposition

Now, can you get the same behaviour *without* providing an example? Write a short instruction prompt (2–3 sentences) telling the model to decompose the question into subquestions.

> 💡 **Tip:** Keep instructions short, clear, and actionable. The model doesn't need a lot of hand-holding!

In [ ]:
prompt = f"""
Instruction: Decompose the following question into a series of subquestions. Each subquestion should be self-contained with all the information necessary to solve it.
Make sure not to decompose more than necessary or have any trivial subquestions - you'll be evaluated on the simplicity, conciseness, and correctness of your decompositions as well as your final answer. 
Once you have all the information you need to answer the question, output the answer.
Q: {test_split_socratic[0]['question']}
A:
"""
message = [{"role": "user", "content": prompt}]
display_model_interaction(message)

---

## 🗺️ Plan-and-Solve Prompting

### What makes this different?

Plan-and-Solve explicitly separates the *planning* phase from the *execution* phase — just like how you'd sketch a rough approach on a whiteboard before diving into code.

**How it works:**
1. **Plan Phase** 🗺️ — The model outlines what steps are needed: "First identify what's asked → then gather given info → compute step by step."
2. **Solve Phase** 🧮 — The model follows its own plan systematically to reach the final answer.

This prevents the model from rushing to compute before it fully understands the problem structure.

**[RUN]** Let's see a few-shot example of Plan-and-Solve in action.

In [ ]:
few_shot_PS = f"""
Follow the format below to solve the question:

Q: James decides to run 3 sprints 3 times a week. He runs 60 meters each sprint. How many total meters does he run a week?
A: Let's first understand the problem, extract relevant variables and their corresponding numerals, and make a complete plan.Then, let's carry out the plan, calculate intermediate variables (pay attention to correct numerical calculation and commonsense), solve the problem step by step, and show the answer.
Output:
Given:
James runs 3 sprints 3 times a week.
Each sprint is 60 meters.
Plan:
We need to calculate the total meters run by James in a week.
Calculation:
Total number of sprints run by James in a week = 3 sprints x 3 times = 9 sprints
Total meters run by James in a week = 9 sprints x 60 meters = 540 meters
Answer:
James runs 540 meters in a week.

Q: In a dance class of 20 students, 20% enrolled in contemporary dance, 25% of the remaining enrolled in jazz dance, and the rest enrolled in hip-hop dance. What percentage of the entire students enrolled in hip-hop dance?
A: Let's first understand the problem, extract relevant variables and their corresponding numerals, and make a complete plan.Then, let's carry out the plan, calculate intermediate variables (pay attention to correct numerical calculation and commonsense), solve the problem step by step, and show the answer.
Output:
Given:
Total number of students = 20
Percentage of students enrolled in contemporary dance = 20%
Percentage of students enrolled in jazz dance = 25%
Plan:
1. Calculate the number of students enrolled in contemporary dance.
2. Calculate the number of students remaining after subtracting the number of students enrolled in
contemporary dance.
3. Calculate the number of students enrolled in jazz dance.
4. Calculate the number of students enrolled in hip-hop dance.
5. Calculate the percentage of students enrolled in hip-hop dance.
Calculation:
1. Number of students enrolled in contemporary dance = 20% of 20 = 20% * 20 = 4
2. Number of students remaining after subtracting the number of students enrolled in contemporary
dance = 20 - 4 = 16
3. Number of students enrolled in jazz dance = 25% of 16 = 25% * 16 = 4
4. Number of students enrolled in hip-hop dance = 16 - 4 = 12
5. Percentage of students enrolled in hip-hop dance = 12/20 * 100% = 60%
Answer:
60% of the entire students enrolled in hip-hop dance.

Test Question:
Q: {test_split_socratic[0]['question']}
A:
"""

In [ ]:
message = [{"role": "user", "content": few_shot_PS}]
display_model_interaction(message)

### 🎯 [TODO] Zero-shot Plan-and-Solve

Now write your own zero-shot instruction that teaches the model to plan before solving. Your instruction should direct the model to: (1) outline what's given, (2) devise a plan, (3) execute it step by step, and (4) show the final answer.

In [ ]:
prompt = f"Instruction: First outline what is given, then devise the whole plan to solve the problem, execute it and show the final answer. Don't solve in intermediate steps.\n\n {test_split_socratic[0]['question']}"
message = [{"role": "user", "content": prompt}]
display_model_interaction(message)

# 🏁 Key Takeaways

Woohoo! You just explored three very different flavours of structured reasoning. Here's what to carry with you:

1. **Zero-shot reasoning is powerful** — You don't always need labelled examples. A well-crafted instruction can teach the model the *format* of reasoning on the fly. This is incredibly useful when you have no training examples to draw from.

2. **Reasoning techniques boost accuracy *and* interpretability** — All three approaches (ToT, Subquestion Decomposition, Plan-and-Solve) don't just improve answers; they also make the model's reasoning *visible*, which is useful for debugging and trust.

3. **LLMs reason better when structured** — Just like humans, LLMs benefit from being told to explore, decompose, or plan before answering. The model isn't "smarter" — it's just given a better framework to operate within.

> 🔮 **Next up:** In notebook 3, you'll see how to go one step further — using the model to *automatically discover better prompts* through OPRO!